## Example 1: Parsing regex patterns with parse_pattern()

In [ ]:
import interegular

# Example 1: Parsing regex patterns with parse_pattern()
print("Example 1: Parsing regex patterns")

# Parse different regex patterns
patterns = [
    r'[a-z]+',
    r'\d{3}-\d{4}',
    r'(hello|world)',
    r'[A-Z][a-z]*',
    r'colou?r'
]

print("Parsing regex patterns:")
for pattern_str in patterns:
    try:
        parsed = interegular.parse_pattern(pattern_str)
        print(f"  '{pattern_str}' -> {type(parsed).__name__}")
        print(f"    Options: {len(parsed.options)}")
        
        # Get alphabet of pattern with flags
        alphabet = parsed.get_alphabet(interegular.REFlags(0))
        print(f"    Alphabet size: {len(alphabet)}")
        
    except Exception as e:
        print(f"  '{pattern_str}' -> Error: {e}")
        raise

# Each parsed pattern compiles to an FSM that accepts exactly what the regex matches
def accepts(pattern_str, text):
    return interegular.parse_pattern(pattern_str).to_fsm().accepts(text)

assert accepts(r'[a-z]+', 'hello') and not accepts(r'[a-z]+', 'Hello')
assert accepts(r'\d{3}-\d{4}', '555-1234') and not accepts(r'\d{3}-\d{4}', '5551234')
assert accepts(r'colou?r', 'color') and accepts(r'colou?r', 'colour')
assert len(interegular.parse_pattern(r'(hello|world)').options) == 2

## Example 2: Comparing regex patterns with compare_regexes()

In [ ]:
import interegular

# Example 2: Comparing regex patterns with compare_regexes()
print("Example 2: Comparing regex patterns")

# Compare pairs of regex patterns (fix escape sequence warning)
pattern_pairs = [
    ('[a-z]+', '[a-zA-Z]+'),
    (r'\d+', '[0-9]+'),
    ('colou?r', 'colo(u|ur)r'),
    ('(cat|dog)', '(dog|cat)'),
    ('[A-Z][a-z]*', '[A-Z][a-zA-Z]*')
]

print("Comparing regex patterns:")
for pattern1, pattern2 in pattern_pairs:
    try:
        comparison = interegular.compare_regexes(pattern1, pattern2)
        
        print(f"  '{pattern1}' vs '{pattern2}':")
        
        # compare_regexes yields every pair of patterns that can match a common string
        results = list(comparison)
        print(f"    Overlapping pairs: {results}")
        assert results == [(pattern1, pattern2)], results

    except Exception as e:
        print(f"  Error comparing: {e}")
        raise

# Patterns with no string in common produce no pairs
assert list(interegular.compare_regexes('[a-z]+', '[0-9]+')) == []
assert list(interegular.compare_regexes('[a-z]+', '[0-9]+', '[a-c]x')) == [('[a-z]+', '[a-c]x')]

## Example 3: Converting patterns to FSM with to_fsm()

In [ ]:
import interegular

# Example 3: Converting patterns to FSM with to_fsm()
print("Example 3: Converting patterns to finite state machines")

# Create and convert patterns to FSM
patterns = ['abc', '[a-c]+', 'a*b', '(ab|cd)', 'x?y']

print("Converting patterns to FSM:")
for pattern_str in patterns:
    try:
        pattern = interegular.parse_pattern(pattern_str)
        fsm = pattern.to_fsm()
        
        print(f"  '{pattern_str}':")
        print(f"    FSM type: {type(fsm).__name__}")
        print(f"    States: {len(fsm.states)}")
        print(f"    Initial state: {fsm.initial}")
        print(f"    Final states: {len(fsm.finals)}")
        print(f"    Alphabet size: {len(fsm.alphabet)}")
        
    except Exception as e:
        print(f"  '{pattern_str}' -> Error: {e}")
        raise

fsms = {p: interegular.parse_pattern(p).to_fsm() for p in patterns}
assert fsms['abc'].accepts('abc') and not fsms['abc'].accepts('ab')
assert fsms['[a-c]+'].accepts('cab') and not fsms['[a-c]+'].accepts('')
assert fsms['a*b'].accepts('b') and fsms['a*b'].accepts('aaab') and not fsms['a*b'].accepts('ba')
assert fsms['(ab|cd)'].accepts('cd') and not fsms['(ab|cd)'].accepts('ad')
assert fsms['x?y'].accepts('y') and fsms['x?y'].accepts('xy')
assert len(fsms['abc'].states) == 4 and len(fsms['abc'].finals) == 1

## Example 4: Pattern simplification with simplify()

In [ ]:
import interegular

# Example 4: Pattern simplification with simplify()
print("Example 4: Pattern simplification")

# Create patterns that can be simplified
complex_patterns = [
    'a|a',        # Redundant alternation
    '(a)*',       # Nested repetition
    'a{1,1}',     # Redundant quantifier
    '(ab|ab)',    # Duplicate alternatives
    'a{0,}',      # Equivalent to a*
]

print("Simplifying complex patterns:")
for pattern_str in complex_patterns:
    try:
        pattern = interegular.parse_pattern(pattern_str)
        simplified = pattern.simplify()
        
        print(f"  '{pattern_str}':")
        print(f"    Original: {len(str(pattern))} chars")
        print(f"    Simplified: {len(str(simplified))} chars")
        print(f"    Same pattern: {pattern is simplified}")
        
        # Check if alphabets are the same (with flags parameter)
        flags = interegular.REFlags(0)
        orig_alphabet = pattern.get_alphabet(flags)
        simp_alphabet = simplified.get_alphabet(flags)
        print(f"    Same alphabet: {orig_alphabet == simp_alphabet}")
        
    except Exception as e:
        print(f"  '{pattern_str}' -> Error: {e}")
        raise

# simplify() never changes the language: each simplified FSM is equivalent to the original
for pattern_str in complex_patterns:
    pattern = interegular.parse_pattern(pattern_str)
    assert pattern.to_fsm().equivalent(pattern.simplify().to_fsm()), pattern_str
assert interegular.parse_pattern('a{0,}').to_fsm().equivalent(interegular.parse_pattern('a*').to_fsm())

## Example 5: Pattern analysis with lengths() and prefix_postfix()

In [ ]:
import interegular

# Example 5: Pattern analysis with lengths() and prefix_postfix()
print("Example 5: Pattern analysis and properties")

# Analyze various patterns
test_patterns = [
    'abc',        # Fixed length
    '[a-z]{3}',   # Fixed length range
    'hello+',     # Variable length
    'a*',         # Zero or more
    'test|word'   # Alternative fixed lengths
]

# (min, max) length of every match; None = unbounded
expected_lengths = {'abc': (3, 3), '[a-z]{3}': (3, 3), 'hello+': (5, None), 'a*': (0, None), 'test|word': (4, 4)}

print("Analyzing pattern properties:")
for pattern_str in test_patterns:
    pattern = interegular.parse_pattern(pattern_str)
    
    print(f"  '{pattern_str}':")
    
    # Possible lengths: `lengths` is a plain (min, max) tuple attribute in interegular 0.3, not a method
    lengths_tuple = pattern.lengths
    print(f"    Possible lengths: {lengths_tuple}")
    assert lengths_tuple == expected_lengths[pattern_str], (pattern_str, lengths_tuple)
    
    # Prefix/postfix analysis: also a tuple attribute
    prefix_postfix = pattern.prefix_postfix
    print(f"    Prefix/Postfix: {len(str(prefix_postfix))} chars")
    
    # Check alphabet (with flags parameter)
    alphabet = pattern.get_alphabet(interegular.REFlags(0))
    sample_chars = list(alphabet)[:5] if alphabet else []
    print(f"    Sample alphabet chars: {sample_chars}")

print("Pattern analysis completed successfully")